# 第89课 · RAG 完整流水线 — 文档切片（Document Chunking） + TF-IDF 检索 + 提示词（Prompt）构建 + 生成

**学习目标**
- 把 L88 的 TF-IDF 检索器组装进完整 RAG 流水线
- 实现 chunk_text() 文档切片
- 实现 format_rag_prompt() 提示词模板
- 跑通：问题 → 检索 → 填充上下文 → 生成回答

← **上一课**　[L88 · TF-IDF 检索从零实现](L88_tfidf_retrieval.ipynb)

> 上节课学习了 **TF-IDF 检索从零实现**：词频逆文档频率，纯 NumPy 向量检索。  
> 本课将探讨 **RAG 完整流水线**。

## 有没有想过：ChatGPT 是怎么知道它不知道的事情的？

大型语言模型的知识截止于训练数据——它无法回答"你们公司的 API 文档里怎么说？"

**RAG（检索增强生成 Retrieval-Augmented Generation）的解法**：

1. 先**检索**（Retrieve）——从你的文档库里找最相关的段落
2. 再**增强**（Augment）——把段落塞进提示词
3. 最后**生成**（Generate）——让 LLM 用这些上下文回答

用图书馆做比喻：不是让图书馆员凭记忆回答，而是先去书架上翻出相关书页，再根据书页内容回答。

```
用户问题
   ↓
[切片] 长文档 → 小 chunk，便于精确匹配
   ↓
[检索] TF-IDF cosine 搜索 → top-k 相关 chunk
   ↓
[增强] 把 chunk 填入提示词模板
   ↓
[生成] LLM 根据上下文回答 + 引用来源
```

**为什么 TF-IDF 已经够用？**
- 不需要 GPU、不需要预训练 embedding 模型
- 关键词匹配（"FFT"、"Nyquist"）对技术文档效果很好
- 整个检索栈可以完全离线运行

本节实现 `chunk_text`（滑动窗口分块）和 `format_rag_prompt`（提示词模板），用 `aurora.llm` 的 TF-IDF 检索器跑通完整 RAG 流水线。

In [ ]:
import numpy as np
from aurora.llm import build_tfidf, cosine_retrieve

In [ ]:
# 知识库（音频 AI 相关，更长的段落）
KNOWLEDGE_BASE = [
    """The Fast Fourier Transform (FFT) is an algorithm that computes the Discrete Fourier Transform
in O(N log N) time. It was developed by Cooley and Tukey in 1965. The FFT works by recursively
splitting the DFT into even and odd sub-problems, known as the Cooley-Tukey radix-2 algorithm.""",

    """The Short-Time Fourier Transform (STFT) applies a windowed FFT to a signal at successive
time steps to produce a spectrogram. It provides a time-frequency representation showing how the
frequency content of a signal changes over time. The window size controls the time-frequency
resolution tradeoff.""",

    """Mel Frequency Cepstral Coefficients (MFCCs) are widely used features in speech recognition.
They are computed by applying a filterbank of triangular filters on the mel scale, taking the
log, and applying a Discrete Cosine Transform. The mel scale approximates the human auditory
system's logarithmic frequency perception.""",

    """CTC (Connectionist Temporal Classification) is a training objective for sequence-to-sequence
models where the alignment between input and output is unknown. It introduces a blank symbol and
sums over all possible alignments using a forward-backward algorithm. CTC is used in speech
recognition models like DeepSpeech.""",

    """Whisper is a speech recognition model from OpenAI trained on 680,000 hours of multilingual
audio. It uses a Transformer encoder-decoder architecture with log-Mel spectrogram inputs.
Whisper can transcribe speech in multiple languages and translate to English.""",

    """The Nyquist-Shannon sampling theorem states that a signal must be sampled at a rate at least
twice its highest frequency component to be reconstructed without aliasing. For example, CD audio
uses 44,100 Hz to capture frequencies up to 22,050 Hz, covering the full range of human hearing.""",
]
print(f'知识库：{len(KNOWLEDGE_BASE)} 个段落')

## ✏️ 任务 1：文档切片 `chunk_text`

长文档直接送入检索会降低精度（一个 chunk 包含太多不同话题）。**滑动窗口切片**将文档切成有重叠的小块，重叠部分保证重要信息不被切断。

**签名**：
```python
def chunk_text(text: str, max_words: int = 80, overlap: int = 10) -> list[str]:
```

**3步实现路线**：

| 步骤 | 操作 | 代码提示 |
|---|---|---|
| 1 | 分词 | `words = text.split()` |
| 2 | 滑动窗口 | `step = max_words - overlap`；从 0 开始每次步进 step |
| 3 | 拼回字符串 | `' '.join(words[i:i+max_words])` 存入列表 |

**验收标准**：
- 短文本（≤max_words 词）→ 返回 `[text]`（单块）
- 200词文本 with max_words=80,overlap=10 → 多块，每块 ≤80词
- 相邻块有 `overlap` 个重叠词

In [ ]:
def chunk_text(text: str, max_words: int = 80, overlap: int = 10) -> list:
    """将长文本切成有重叠的词块。"""
    words = text.split()
    if len(words) <= max_words:
        return [text]
    # ✏️ TODO: 滑动窗口切块
    raise NotImplementedError("TODO")

# 测试
try:
    long_text = " ".join(["word"] * 200)
    chunks = chunk_text(long_text, max_words=80, overlap=10)
    print(f"200 词文本 → {len(chunks)} 个块（max_words=80, overlap=10）")
except NotImplementedError:
    print("⬜ chunk_text 未实现")

# 对知识库分块（带兜底：未实现时用完整段落）
all_chunks = []
for doc in KNOWLEDGE_BASE:
    try:
        all_chunks.extend(chunk_text(doc, max_words=60))
    except NotImplementedError:
        all_chunks.append(doc)  # 兜底：未实现时保留完整段落
print(f"知识库分块后：{len(all_chunks)} 个 chunk")


In [ ]:
try:
    long_text = ' '.join(['word'] * 200)
    chunks = chunk_text(long_text, max_words=80, overlap=10)
    assert chunks is not None, "⬜ chunk_text 未实现"
    assert isinstance(chunks, list), f"应返回列表，得到 {type(chunks)}"
    assert len(chunks) > 1, f"200 词文本应切出多个块，得到 {len(chunks)} 块"
    for i, chunk in enumerate(chunks):
        assert len(chunk.split()) <= 80, f"块 {i} 词数 {len(chunk.split())} 超过 max_words=80"
    short_chunks = chunk_text("hello world", max_words=80)
    assert len(short_chunks) == 1, "短文本应保持为单块"
    print(f'✅ chunk_text 验证通过：200 词 → {len(chunks)} 个块，每块 ≤ 80 词')
except (NotImplementedError, TypeError):
    print('⬜ chunk_text 未实现')

## ✏️ 任务 2：RAG 提示词模板 `format_rag_prompt`

检索到相关段落后，需要把它们**组织成提示词**发给 LLM。提示词格式直接影响生成质量——段落编号让 LLM 能引用来源，固定格式让 LLM 知道从哪里开始生成回答。

**签名**：
```python
def format_rag_prompt(query: str, retrieved_docs: list) -> str:
    # retrieved_docs: [(doc_str, score), ...]
```

**期望输出格式**：
```
Context:
[1] <第一段检索内容>
[2] <第二段检索内容>

Question: <用户问题>

Answer:
```

**2步实现路线**：

| 步骤 | 操作 | 代码提示 |
|---|---|---|
| 1 | 构建 Context 段落 | `f"[{i+1}] {doc}"` for each `(doc, score)` in retrieved_docs |
| 2 | 拼接完整 prompt | `"Context:\n" + paragraphs + "\n\nQuestion: " + query + "\n\nAnswer:"` |

**验收标准**：
- prompt 包含 query 字符串
- prompt 包含所有检索文档内容
- 有 `[1]`, `[2]` 编号标记，有 `Answer:` 结尾

In [ ]:
def format_rag_prompt(query: str, retrieved_docs: list) -> str:
    """将检索到的段落填入提示词模板。

    Expected output format:
    Context:
    [1] <first retrieved passage>
    [2] <second retrieved passage>

    Question: <query>

    Answer:
    """
    # ✏️ TODO: 构建包含上下文的提示词
    raise NotImplementedError("TODO")


In [ ]:
try:
    test_docs = [("FFT is an algorithm that computes DFT.", 0.85),
                 ("STFT applies windowed FFT to a signal.", 0.72)]
    prompt = format_rag_prompt("What is FFT?", test_docs)
    assert prompt is not None, "⬜ format_rag_prompt 未实现"
    assert isinstance(prompt, str), f"应返回字符串，得到 {type(prompt)}"
    assert "What is FFT?" in prompt, "prompt 应包含查询词"
    assert "FFT is an algorithm" in prompt, "prompt 应包含检索到的文档内容"
    assert len(prompt) > 50, f"prompt 过短（{len(prompt)} 字符）"
    print(f'✅ format_rag_prompt 验证通过（prompt 长度 {len(prompt)} 字符）')
except (NotImplementedError, TypeError, NameError):
    print('⬜ format_rag_prompt 未实现')

In [ ]:
# 构建 TF-IDF 索引
matrix, vocab = build_tfidf(all_chunks)
print(f'索引：{matrix.shape[0]} 个 chunk，词汇量 {matrix.shape[1]}')

# RAG 流水线演示
QUERIES = [
    'How does the FFT algorithm work?',
    'What are MFCCs used for?',
    'How does CTC handle alignment in speech recognition?',
]

for query in QUERIES:
    print(f'\n{"="*55}')
    print(f'问题：{query}')
    print('-'*55)
    results = cosine_retrieve(query, matrix, vocab, all_chunks, top_k=2)
    # 打印检索结果
    for doc, score in results:
        print(f'  [{score:.3f}] {doc[:80]}...')
    # 增强 + 生成提示词（format_rag_prompt 未实现时优雅跳过）
    try:
        prompt = format_rag_prompt(query, results)
        print()
        print('生成提示词（前 200 字符）:')
        print(prompt[:200] + '...')
    except (NotImplementedError, TypeError):
        print('  ⬜ format_rag_prompt 未实现 — 跳过提示词展示')

In [ ]:
# 端到端 RAG 验证
try:
    test_query = "How does the FFT algorithm work?"
    results = cosine_retrieve(test_query, matrix, vocab, all_chunks, top_k=3)
    assert results is not None, "⬜ cosine_retrieve 未实现"
    assert isinstance(results, list), f"应返回列表，得到 {type(results)}"
    assert len(results) <= 3, f"top_k=3 但返回了 {len(results)} 个结果"
    for doc, score in results:
        assert isinstance(doc, str), f"文档应为字符串，得到 {type(doc)}"
        assert isinstance(score, float), f"分数应为浮点数，得到 {type(score)}"
        assert 0.0 <= score <= 1.0, f"余弦相似度 {score} 超出 [0, 1] 范围"
    prompt = format_rag_prompt(test_query, results)
    assert isinstance(prompt, str) and len(prompt) > 0, "prompt 应为非空字符串"
    print(f'✅ 端到端 RAG 验证通过：检索到 {len(results)} 个片段，prompt 长度 {len(prompt)} 字符')
except (NotImplementedError, TypeError, NameError):
    print('⬜ 未实现')

In [ ]:
# 可选：接入真实 LLM 生成回答
try:
    from transformers import pipeline
    print('（如果已安装 transformers + 模型，可在此生成真实回答）')
    # generator = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B')
    # answer = generator(prompt, max_new_tokens=100)[0]['generated_text']
except ImportError:
    print('transformers 未安装 — RAG 检索部分（无需 GPU）已完整运行 ✅')

## 本课收束

| 组件 | 技术 | 依赖 |
|---|---|---|
| 文档切片 | 滑动窗口 | 纯 Python |
| 索引构建 | TF-IDF | aurora.llm + NumPy |
| 检索 | 余弦相似度（Cosine Similarity） k-NN | aurora.llm + NumPy |
| 提示词模板 | 字符串格式化 | 纯 Python |
| 生成 | LLM（可选）| transformers（可选）|

**关键洞察**：RAG 的检索质量决定了生成质量。好的检索 > 大的模型。

下一步 L90：对话式 RAG——会话记忆（Conversation Memory）、来源归因与 Podcast Q&A 流水线。

## ✏️ 闭卷推导检查格 — RAG 数据流图

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：画出完整 RAG pipeline 的数据流图，标注每一步的输入/输出类型：

| 步骤 | 操作 | 输入 | 输出 |
|------|------|------|------|
| 1 | Chunking | 原始文档 (str) | List[str]（块列表） |
| 2 | TF-IDF 索引构建 | List[str] | ___ |
| 3 | Query 向量化 | 用户问题 (str) | ___ |
| 4 | Cosine 检索 | ___ | Top-k 文档块 |
| 5 | Prompt 构建 | 问题 + Top-k 块 | ___ |
| 6 | LLM 生成 | ___ | 回答 (str) |

并说明：为什么 TF-IDF 检索不需要 embedding 模型？

（在此处填表...）

In [ ]:
# 验证：RAG 检索 smoke test（无 LLM 步骤，只测 aurora.llm 检索 API）
from aurora.llm import build_tfidf, cosine_retrieve

docs = [
    "Aurora 是一个从零构建的音频 AI 系统",
    "FFT 全称快速傅里叶变换，时间复杂度 O(N log N)",
    "MFCC 是语音识别最常用的特征，包含 DCT 去相关步骤",
    "Whisper 由 OpenAI 发布，使用 Transformer 架构",
    "KV-Cache 将推理投影总代价从 O(T²) 降至 O(T)",
]
sm_matrix, sm_vocab = build_tfidf(docs)
query = "FFT 的时间复杂度是多少"
results = cosine_retrieve(query, sm_matrix, sm_vocab, docs, top_k=2)
assert len(results) == 2, f"期望 2 个结果，得到 {len(results)}"
assert any("FFT" in doc for doc, _ in results), "Top-2 结果应包含 FFT 相关文档"
print("检索结果：")
for i, (doc, score) in enumerate(results, 1):
    print(f"  {i}. [{score:.3f}] {doc[:50]}...")
print("✅ RAG 检索 smoke test 通过")

In [ ]:
# ✏️ 本课自评
l89_review = {
    "rag_pipeline_map":          None,  # 能画出RAG四步数据流图（切片/检索/增强/生成）？True/False
    "chunk_text_impl":           None,  # chunk_text()滑动窗口实现正确，checker通过？True/False
    "format_prompt_impl":        None,  # format_rag_prompt()格式正确（编号+Answer:）？True/False
    "end_to_end_passed":         None,  # 端到端RAG smoke test通过（cosine_retrieve+prompt）？True/False
    "whiteboard_passed":         None,  # 白板挑战RAG数据流表格填写通关？True/False
}

unfilled = [k for k, v in l89_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l89_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L89 全部通关！进入 L90：对话式 RAG')

---

→ **下一课**　[L90 · 对话式 RAG](L90_agent.ipynb)

> 下节课将学习 **对话式 RAG**：会话记忆、来源归因与 Podcast Q&A 流水线。